In [0]:
from pyspark.sql import Row, functions as F, types as T
from delta.tables import DeltaTable
from datetime import datetime, timezone
import requests
import json
import time
import logging
import uuid
import math
from concurrent.futures import ThreadPoolExecutor, as_completed

In [0]:
# ---------------------------------------------------------------------------
# Widgets
# ---------------------------------------------------------------------------
dbutils.widgets.text("env", "dev", "Environment")
dbutils.widgets.text("is_full_refresh", "False", "Full Refresh (True/False)")

# ---------------------------------------------------------------------------
# Logging Configuration
# ---------------------------------------------------------------------------
NOTEBOOK_NAME    = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().split("/")[-1]

log              = logging.getLogger(NOTEBOOK_NAME)
log.setLevel(logging.INFO)
if not log.handlers:
    _handler     = logging.StreamHandler()
    _handler.setFormatter(
        logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
    )
    log.addHandler(_handler)

# ---------------------------------------------------------------------------
# Dynamic Configuration
# ---------------------------------------------------------------------------
env         = dbutils.widgets.get("env").strip()
# env         = "dev" 
CATALOG          = f"{env}_corpdatav2"
BRONZE_DB        = f"{CATALOG}.bronze"
SILVER_DB        = f"{CATALOG}.silver"
AUDIT_CATALOG    = f"{env}_audit"
LOGS             = f"{AUDIT_CATALOG}.logs"

LOGS_TABLE       = f"{LOGS}.{NOTEBOOK_NAME}"
TARGET_BRONZE    = f"{BRONZE_DB}.AXS_API_AlphalinerVessel"
TARGET_SILVER    = f"{SILVER_DB}.AXS_API_AlphalinerVessel"

COUNTRY_TABLE    = f"{BRONZE_DB}.dw_md_country" 

AXS_API_BASE = "https://webservicesv5.axsmarine.com/rest/vessel/4/liner"

AXS_API_TOKEN   = dbutils.secrets.get(scope="azure-scope", key="adb-alphaliner-apim")
AXS_API_STEP = 200
AXS_API_PAGE = 1
AXS_DEFAULT_DATE = "1900-02-09 12:15:03"

PARAMS = {
    "token": AXS_API_TOKEN,
    "step" : AXS_API_STEP,
    "page" : AXS_API_PAGE,          # overridden per-page in run_pipeline()
    # modifiedDate is injected at runtime from the last run date
    # requests auto-encodes it in the URL as: YYYY-MM-DD%20HH%3AMM%3ASS
}
# ---------------------------------------------------------------------------
# API Response Keys
# ---------------------------------------------------------------------------
AXS_API_TOTAL_KEY = "totalCount"     # JSON key: total record count in response
AXS_API_DATA_KEY  = "results"        # JSON key: records array in response
# ---------------------------------------------------------------------------
# Pipeline Settings
# ---------------------------------------------------------------------------
MAX_RETRIES        = 3
CONCURRENT_WORKERS = 5             # max parallel threads for page fetching
BATCH_RUN_ID     = str(uuid.uuid4())
FULL_REFRESH     = dbutils.widgets.get("is_full_refresh").strip().lower() == "true"

log.info(f"Environment: {env}")
log.info(f"Batch Run ID: {BATCH_RUN_ID}")
log.info(f"Catalog: {CATALOG}")
log.info(f"Max Retries: {MAX_RETRIES}")

In [0]:
audit_schema = T.StructType([
    T.StructField("EventID", T.StringType(), True),
    T.StructField("batch_run_id", T.StringType(), True),
    T.StructField("api_type", T.StringType(), True),
    T.StructField("http_method", T.StringType(), True),
    T.StructField("api_url", T.StringType(), True),
    T.StructField("input_value", T.StringType(), True),
    T.StructField("request_body", T.StringType(), True),
    T.StructField("attempt_number", T.IntegerType(), True),
    T.StructField("request_timestamp", T.TimestampType(), True),
    T.StructField("response_code", T.IntegerType(), True),
    T.StructField("response_body", T.StringType(), True),
    T.StructField("duration_ms", T.LongType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("error_message", T.StringType(), True),
    T.StructField("execution_context", T.StringType(), True),
])

bronze_schema = T.StructType([
    T.StructField("Raw_data", T.StringType(), True),
    T.StructField("Api_metadata", T.StringType(), True),
    T.StructField("_is_normalized", T.BooleanType(), True),
    T.StructField("_Batch_run_id", T.StringType(), True),
    T.StructField("_ingestion_ts", T.TimestampType(), True),
])
_api_vessel_schema = T.StructType([
    T.StructField("vesselBeam", T.FloatType(), True),
    T.StructField("vesselTeu", T.IntegerType(), True),
    T.StructField("vesselBuiltYear", T.IntegerType(), True),
    T.StructField("vesselDwt", T.IntegerType(), True),
    T.StructField("vesselFlag", T.StringType(), True),
    T.StructField("vesselTeu14", T.IntegerType(), True),
    T.StructField("vesselReefPlugsNumber", T.IntegerType(), True),
    T.StructField("vesselSpeedLaden", T.FloatType(), True),
    T.StructField("vesselConsumption", T.FloatType(), True),
    T.StructField("vesselCells", T.StringType(), True),
    T.StructField("vesselGeared", T.StringType(), True),
    T.StructField("vesselIceLevel", T.StringType(), True),
    T.StructField("vesselShipowner", T.StringType(), True),
    T.StructField("vesselCommercialStatus", T.StringType(), True),
    T.StructField("vesselTOP", T.StringType(), True),
    T.StructField("DualFuel", T.StringType(), True),
    T.StructField("engineModel", T.StringType(), True),
    T.StructField("engineDesign", T.StringType(), True),
    T.StructField("vesselDesign", T.StringType(), True),
    T.StructField("vesselRoute", T.StringType(), True),
    T.StructField("vesselOperator", T.StringType(), True),
    T.StructField("vesselBuilder", T.StringType(), True),
    T.StructField("vesselLoa", T.FloatType(), True),
    T.StructField("vesselDraft", T.FloatType(), True),
    T.StructField("vesselImo", T.IntegerType(), True),
    T.StructField("Scrubbers", T.StringType(), True),
    T.StructField("vesselGrt", T.IntegerType(), True),
    T.StructField("vesselName", T.StringType(), True),
    T.StructField("vesselType", T.StringType(), True),
    T.StructField("vesselTypeShort", T.StringType(), True),
])

_api_response_schema = T.StructType([
    T.StructField("totalCount", T.IntegerType(), True),
    T.StructField("results", T.ArrayType(_api_vessel_schema), True),
])

_api_metadata_schema = T.StructType([
    T.StructField("Status", T.StringType(), True),
    T.StructField("page_no", T.IntegerType(), True),
])

# ---------------------------------------------------------------------------
# Silver Schema (from dev_corpdata.bronze.axs_alphalinervesselapi EXCEPT ETL_LOAD_DTM)
# ---------------------------------------------------------------------------
silver_schema = T.StructType([
    T.StructField("file_date", T.StringType(), True),
    T.StructField("st", T.StringType(), True),
    T.StructField("name", T.StringType(), True),
    T.StructField("teu", T.IntegerType(), True),
    T.StructField("built", T.IntegerType(), True),
    T.StructField("dwt", T.IntegerType(), True),
    T.StructField("flag", T.StringType(), True),
    T.StructField("at14_t", T.IntegerType(), True),
    T.StructField("plugs", T.IntegerType(), True),
    T.StructField("speed", T.FloatType(), True),
    T.StructField("cns", T.FloatType(), True),
    T.StructField("c", T.IntegerType(), True),
    T.StructField("g", T.IntegerType(), True),
    T.StructField("ice", T.StringType(), True),
    T.StructField("managing_owner", T.StringType(), True),
    T.StructField("t", T.StringType(), True),
    T.StructField("open_position", T.StringType(), True),
    T.StructField("dual_fuel", T.StringType(), True),
    T.StructField("main_engine", T.StringType(), True),
    T.StructField("open_start", T.StringType(), True),
    T.StructField("open_end", T.StringType(), True),
    T.StructField("design_class", T.StringType(), True),
    T.StructField("route", T.StringType(), True),
    T.StructField("pos_amend", T.StringType(), True),
    T.StructField("operator", T.StringType(), True),
    T.StructField("ship_yard", T.StringType(), True),
    T.StructField("beam", T.FloatType(), True),
    T.StructField("loa", T.FloatType(), True),
    T.StructField("draft", T.FloatType(), True),
    T.StructField("owned_chartered", T.StringType(), True),
    T.StructField("imo", T.IntegerType(), True),
    T.StructField("scrubber", T.IntegerType(), True),
    T.StructField("gt", T.IntegerType(), True),
    T.StructField("FileName", T.StringType(), True),
    T.StructField("vesselType", T.StringType(), True),
    T.StructField("APIPageNum", T.StringType(), True),
])

In [0]:
# ---------------------------------------------------------------------------
# Helper: Get Last Run Date from Logs
# ---------------------------------------------------------------------------
def get_last_run_date(logs_table: str, default_date: str, full_refresh: bool = False) -> datetime:
    """
    Return the date to use as modifiedDate for the API call.
    - full_refresh=True  : skip logs query; return default_date (reprocess everything).
    - full_refresh=False : query logs for the latest SUCCESS timestamp; fall back to
                           default_date when the table is absent or has no SUCCESS rows.
    """
    if full_refresh:
        dt = datetime.strptime(default_date, "%Y-%m-%d %H:%M:%S")
        log.info(f"Full refresh enabled – using default date: {dt}")
        return dt
    # Use tableExists() so lazy transforms are NOT inside a try/except (suppresses SCPAP005).
    # Only the action (collect) is inside try to catch query-time errors.
    if spark.catalog.tableExists(logs_table):
        df = (spark.table(logs_table)
                   .filter(F.col("status") == "SUCCESS")
                   .agg(F.max("request_timestamp").alias("last_ts")))
        try:
            last_ts = df.collect()[0]["last_ts"]   # plan is analysed here; errors surface at collect()
            if last_ts is not None:
                log.info(f"Last successful run from logs: {last_ts}")
                return last_ts
        except Exception as exc:
            log.warning(f"Could not query {logs_table}: {exc}. Falling back to default.")
    else:
        log.warning(f"Logs table {logs_table} not found. Falling back to default.")
    dt = datetime.strptime(default_date, "%Y-%m-%d %H:%M:%S")
    log.info(f"Using default date: {dt}")
    return dt


# ---------------------------------------------------------------------------
# Helper: Call API with Retry
# ---------------------------------------------------------------------------
def call_api(url: str, params: dict, max_retries: int = 3) -> dict:
    """
    HTTP GET with exponential-backoff retry.
    Returns:
        {
          "success"  : bool,
          "data"     : dict | None,
          "attempts" : list[dict]    # one entry per attempt
        }
    Status per attempt:
        SUCCESS  – HTTP 200-204
        RETRY    – failed but attempts remain
        FAILED   – last attempt exhausted
    """
    attempts   = []
    attempt    = 0
    _429_count = 0          # guard: prevent infinite loop if server rate-limits indefinitely
    while attempt < max_retries:
        req_ts  = datetime.now(timezone.utc)
        t_start = time.time()
        try:
            resp        = requests.get(url, params=params, timeout=30)
            duration_ms = int((time.time() - t_start) * 1000)

            # 429 Rate-Limited: honour Retry-After; do NOT count as a retry attempt
            if resp.status_code == 429:
                _429_count += 1
                if _429_count > max_retries * 3:   # guard: too many consecutive 429s
                    attempt   += 1
                    att_status = "RETRY" if attempt < max_retries else "FAILED"
                    attempts.append({
                        "attempt_number"   : attempt,
                        "request_timestamp": req_ts,
                        "response_code"    : 429,
                        "response_body"    : resp.text[:65535],
                        "duration_ms"      : duration_ms,
                        "status"           : att_status,
                        "error_message"    : f"Aborted: {_429_count} consecutive 429 responses (threshold={max_retries * 3})",
                    })
                    log.error(f"429 threshold exceeded ({_429_count} consecutive) – treating as failed attempt.")
                    break
                try:
                    wait_secs = int(resp.headers.get("Retry-After", 5))
                except ValueError:
                    wait_secs = 5
                log.warning(f"429 Rate-Limited – waiting {wait_secs}s (Retry-After) ...")
                time.sleep(wait_secs)
                continue                        # loop again WITHOUT incrementing attempt

            is_ok      = 200 <= resp.status_code <= 204
            attempt   += 1
            att_status = "SUCCESS" if is_ok else ("RETRY" if attempt < max_retries else "FAILED")
            attempts.append({
                "attempt_number"   : attempt,
                "request_timestamp": req_ts,
                "response_code"    : resp.status_code,
                "response_body"    : resp.text[:65535],
                "duration_ms"      : duration_ms,
                "status"           : att_status,
                "error_message"    : None if is_ok else f"HTTP {resp.status_code}: {resp.reason}",
            })

            if is_ok:
                return {"success": True, "data": resp.json(), "attempts": attempts}

        except Exception as exc:
            duration_ms = int((time.time() - t_start) * 1000)
            attempt    += 1
            attempts.append({
                "attempt_number"   : attempt,
                "request_timestamp": req_ts,
                "response_code"    : -1,
                "response_body"    : None,
                "duration_ms"      : duration_ms,
                "status"           : "RETRY" if attempt < max_retries else "FAILED",
                "error_message"    : str(exc),
            })

        if attempt < max_retries:
            wait = 2 ** attempt
            log.warning(f"Attempt {attempt}/{max_retries} failed – retrying in {wait}s ...")
            time.sleep(wait)

    return {"success": False, "data": None, "attempts": attempts}


# ---------------------------------------------------------------------------
# Helper: Log Audit
# ---------------------------------------------------------------------------
def log_audit(
    result           : dict,
    logs_table       : str,
    audit_schema,
    batch_run_id     : str,
    api_url          : str,
    input_value      : str,
    request_body     : str,
    execution_context: str,
) -> None:
    """
    Persist every attempt in result['attempts'] to the audit/logs Delta table.
    One row per retry attempt – covers RETRY, SUCCESS, and FAILED statuses.
    """
    rows = [
        Row(
            EventID           = str(uuid.uuid4()),
            batch_run_id      = batch_run_id,
            api_type          = "REST",
            http_method       = "GET",
            api_url           = api_url,
            input_value       = input_value,
            request_body      = request_body,
            attempt_number    = a["attempt_number"],
            request_timestamp = a["request_timestamp"],
            response_code     = a["response_code"],
            response_body     = a["response_body"],
            duration_ms       = a["duration_ms"],
            status            = a["status"],
            error_message     = a["error_message"],
            execution_context = execution_context,
        )
        for a in result["attempts"]
    ]
    (
        spark.createDataFrame(rows, schema=audit_schema)
             .write.format("delta")
             .mode("append")
             .option("mergeSchema", "true")
             .saveAsTable(logs_table)
    )
    log.info(f"Audit: {len(rows)} row(s) -> {logs_table}  [final status={result['attempts'][-1]['status']}]")


# ---------------------------------------------------------------------------
# Helper: Append to Bronze
# ---------------------------------------------------------------------------
def append_bronze(
    bronze_table : str,
    bronze_schema,
    raw_data     : dict,
    last_attempt : dict,
    batch_run_id : str,
    page_no      : int = 1,
) -> None:
    """
    Write one API-page response as a single Bronze row.
    Raw_data     = full page JSON string.
    Api_metadata = structured call summary (Status, HttpStatus, Timestamp, ErrorMessage).
    _is_normalized is always False at ingest time.
    """
    metadata = json.dumps({
        "Status"      : last_attempt["status"],
        "ErrorMessage": last_attempt["error_message"],
        "HttpStatus"  : last_attempt["response_code"],
        "Timestamp"   : last_attempt["request_timestamp"].isoformat(),
        "page_no"     : page_no,
    })
    row = Row(
        Raw_data       = json.dumps(raw_data),
        Api_metadata   = metadata,
        _is_normalized = False,
        _Batch_run_id  = batch_run_id,
        _ingestion_ts  = datetime.now(timezone.utc),
    )
    (
        spark.createDataFrame([row], schema=bronze_schema)
             .write.format("delta")
             .mode("append")
             .option("mergeSchema", "true")
             .saveAsTable(bronze_table)
    )
    log.info(f"Bronze: 1 page appended -> {bronze_table}")


# ---------------------------------------------------------------------------
# Batch Write: Audit Logs (all pages, single Delta append)
# ---------------------------------------------------------------------------
def batch_write_logs(fetched: list, modified_date: str) -> None:
    """
    Write audit rows for every attempt across ALL fetched pages in one Delta append.
    fetched : list of (page, result, params_used) tuples collected by run_pipeline.
    One audit row per API attempt per page (covers RETRY, SUCCESS, FAILED).
    """
    rows = []
    for _, result, params_used in fetched:
        request_body = json.dumps(params_used)
        for a in result["attempts"]:
            rows.append(Row(
                EventID           = str(uuid.uuid4()),
                batch_run_id      = BATCH_RUN_ID,
                api_type          = "REST",
                http_method       = "GET",
                api_url           = AXS_API_BASE,
                input_value       = modified_date,
                request_body      = request_body,
                attempt_number    = a["attempt_number"],
                request_timestamp = a["request_timestamp"],
                response_code     = a["response_code"],
                response_body     = a["response_body"],
                duration_ms       = a["duration_ms"],
                status            = a["status"],
                error_message     = a["error_message"],
                execution_context = NOTEBOOK_NAME,
            ))
    (
        spark.createDataFrame(rows, schema=audit_schema)
             .write.format("delta")
             .mode("append")
             .option("mergeSchema", "true")
             .saveAsTable(LOGS_TABLE)
    )
    log.info(f"Audit batch: {len(rows)} row(s) from {len(fetched)} page(s) -> {LOGS_TABLE}")


# ---------------------------------------------------------------------------
# Batch Write: Bronze (all successful pages, single Delta append)
# ---------------------------------------------------------------------------
def batch_write_bronze(fetched: list) -> None:
    """
    Write one bronze row per successful non-empty page in one Delta append.
    fetched : list of (page, result, params_used) tuples collected by run_pipeline.
    FAILED or empty pages are skipped (captured in audit only).
    """
    rows = []
    for page, result, _ in fetched:
        if result["success"] and result["data"] and result["data"].get(AXS_API_DATA_KEY):
            last_attempt = result["attempts"][-1]
            metadata = json.dumps({
                "Status"      : last_attempt["status"],
                "ErrorMessage": last_attempt["error_message"],
                "HttpStatus"  : last_attempt["response_code"],
                "Timestamp"   : last_attempt["request_timestamp"].isoformat(),
                "page_no"     : page,
            })
            rows.append(Row(
                Raw_data       = json.dumps(result["data"]),
                Api_metadata   = metadata,
                _is_normalized = False,
                _Batch_run_id  = BATCH_RUN_ID,
                _ingestion_ts  = datetime.now(timezone.utc),
            ))
    if not rows:
        log.warning("Bronze batch: no successful pages with data to write.")
        return
    (
        spark.createDataFrame(rows, schema=bronze_schema)
             .write.format("delta")
             .mode("append")
             .option("mergeSchema", "true")
             .saveAsTable(TARGET_BRONZE)
    )
    log.info(f"Bronze batch: {len(rows)} page(s) -> {TARGET_BRONZE}")


# ---------------------------------------------------------------------------
# Worker: Fetch Single Page (used by run_concurrent)
# ---------------------------------------------------------------------------
def _fetch_page(page: int, modified_date: str) -> tuple:
    """
    Fetch a single API page. No writes – caller handles batch flush after all pages.
    Returns (page, result, params_used).
    """
    pg_params = {**PARAMS, "modifiedDate": modified_date, "page": page}
    log.info(f"Page {page}: fetching ...")
    result    = call_api(AXS_API_BASE, pg_params, MAX_RETRIES)
    last_att  = result["attempts"][-1]
    rec_count = len(result["data"].get(AXS_API_DATA_KEY, [])) if result["data"] else 0
    log.info(f"Page {page}: done – status={last_att['status']}, http={last_att['response_code']}, records={rec_count}")
    if not result["success"]:
        log.error(f"Page {page} FAILED after {MAX_RETRIES} retries.")
    return page, result, pg_params


# ---------------------------------------------------------------------------
# Helper: Concurrent Page Fetcher
# ---------------------------------------------------------------------------
def run_concurrent(pages: list, modified_date: str) -> list:
    """
    Fetch pages in parallel. No writes – caller handles batch flush after all pages.
    Returns sorted list of (page, result, params_used) tuples.
    """
    page_results = []
    with ThreadPoolExecutor(max_workers=CONCURRENT_WORKERS) as executor:
        futures = {executor.submit(_fetch_page, page, modified_date): page for page in pages}
        for future in as_completed(futures):
            orig_page = futures[future]
            try:
                page_results.append(future.result())
            except Exception as exc:
                log.error(f"Page {orig_page} raised unexpected exception: {exc}")
                pg_params = {**PARAMS, "modifiedDate": modified_date, "page": orig_page}
                page_results.append((orig_page, {
                    "success" : False, "data": None,
                    "attempts": [{
                        "attempt_number"   : 1,
                        "request_timestamp": datetime.now(timezone.utc),
                        "response_code"    : -1,
                        "response_body"    : None,
                        "duration_ms"      : 0,
                        "status"           : "FAILED",
                        "error_message"    : str(exc),
                    }],
                }, pg_params))
    return sorted(page_results, key=lambda t: t[0])


# ---------------------------------------------------------------------------
# Pipeline Orchestration: Paginated Ingestion
# ---------------------------------------------------------------------------
def run_pipeline(modified_date: str) -> int:
    """
    Full paginated ingestion flow:
      Page 1  -> call API to derive total record count -> compute total pages.
      Page 1+ -> audit every attempt; append non-empty SUCCESS pages to Bronze.
    Raises RuntimeError if any page exhausts all retries.
    Returns total number of pages processed.
    """
    all_fetched = []   # accumulates (page, result, params_used) for every page

    # -- Page 1: baseline call (app-level retry on total=-1 + short body) ----
    p1_params = {**PARAMS, "modifiedDate": modified_date}

    for p1_retry in range(MAX_RETRIES):
        log.info(f"Page 1: fetching ... (app-attempt {p1_retry + 1}/{MAX_RETRIES})")
        result    = call_api(AXS_API_BASE, p1_params, MAX_RETRIES)
        last_att  = result["attempts"][-1]
        rec_count = len(result["data"].get(AXS_API_DATA_KEY, [])) if result["data"] else 0
        log.info(f"Page 1: done – status={last_att['status']}, http={last_att['response_code']}, records={rec_count}")

        if not result["success"]:
            all_fetched.append((1, result, p1_params))
            batch_write_logs(all_fetched, modified_date)
            raise RuntimeError(
                f"Page 1 failed after {MAX_RETRIES} retries. "
                f"Batch={BATCH_RUN_ID}. See: {LOGS_TABLE}"
            )

        total_count = result["data"].get(AXS_API_TOTAL_KEY, 0)
        body_len    = len(result["attempts"][-1]["response_body"] or "")

        # App-level retry: total=-1 with short body — sentinel/error response.
        if total_count == -1 and body_len < 200:
            if p1_retry < MAX_RETRIES - 1:
                log.warning(
                    f"Page 1: total=-1, body={body_len} chars – "
                    f"retrying ({p1_retry + 1}/{MAX_RETRIES}) ..."
                )
                time.sleep(2 ** (p1_retry + 1))
                continue
            all_fetched.append((1, result, p1_params))
            batch_write_logs(all_fetched, modified_date)
            raise RuntimeError(
                f"Page 1 returned total=-1 with short body ({body_len} chars) "
                f"after {MAX_RETRIES} retries. Batch={BATCH_RUN_ID}. See: {LOGS_TABLE}"
            )
        break   # valid response received

    all_fetched.append((1, result, p1_params))
    total_pages = math.ceil(total_count / AXS_API_STEP) if total_count > 0 else 1
    log.info(f"Total records: {total_count} | Total pages: {total_pages}")

    # -- Pages 2 ... N (concurrent fetch, no writes yet) --------------------
    if total_pages > 1:
        all_fetched += run_concurrent(list(range(2, total_pages + 1)), modified_date)

    # -- Batch flush: all logs then all bronze (single write each) -----------
    log.info(f"All {len(all_fetched)} page(s) fetched – flushing to logs and bronze ...")
    batch_write_logs(all_fetched, modified_date)
    batch_write_bronze(all_fetched)

    # -- Final gate ----------------------------------------------------------
    failed_pages = sorted(p for p, r, _ in all_fetched if not r["success"])
    if failed_pages:
        raise RuntimeError(
            f"Pipeline completed with failures on page(s): {failed_pages}. "
            f"Batch={BATCH_RUN_ID}. See: {LOGS_TABLE}"
        )

    log.info(f"Pipeline complete | pages={total_pages} | batch={BATCH_RUN_ID}")
    return total_pages

In [0]:
# ---------------------------------------------------------------------------
# Normalize Bronze → Silver
# ---------------------------------------------------------------------------
def normalize_to_silver() -> None:
    """
    Flatten un-normalized bronze rows (latest batch only) into the silver table.
    Marks processed bronze rows as _is_normalized = True after successful write.
    """
    # 1. Identify the latest un-normalized batch
    latest_batch = (
        spark.table(TARGET_BRONZE)
        .filter(F.col("_is_normalized") == False)
        .orderBy(F.col("_ingestion_ts").desc())
        .select("_Batch_run_id")
        .first()
    )
    if latest_batch is None:
        log.info("No un-normalized bronze rows found. Skipping normalization.")
        return

    batch_id = latest_batch["_Batch_run_id"]
    log.info(f"Normalizing batch: {batch_id}")

    # 2. Read bronze rows for that batch
    bronze_df = (
        spark.table(TARGET_BRONZE)
        .filter(
            (F.col("_is_normalized") == False) &
            (F.col("_Batch_run_id") == batch_id)
        )
    )

    # 3. Parse Raw_data and Api_metadata, explode results array
    exploded_df = (
        bronze_df
        .withColumn("parsed", F.from_json("Raw_data", _api_response_schema))
        .withColumn("meta", F.from_json("Api_metadata", _api_metadata_schema))
        .withColumn("total_pages", F.ceil(F.col("parsed.totalCount") / F.lit(AXS_API_STEP)))
        .withColumn("vessel", F.explode("parsed.results"))
    )

    country_df = spark.table(COUNTRY_TABLE).select("flag", "alphaliner_flag")

    # 3b. Left join exploded_df with country_df: vessel.vesselFlag -> country_df.flag
    exploded_df = exploded_df.join(
        country_df,
        F.col("vessel.vesselFlag") == F.col("flag"),
        "left",
    )

    # 4. Map API fields → silver schema columns
    silver_df = exploded_df.select(
        F.date_format(F.col("_ingestion_ts"), "yyyy-MM-dd").alias("file_date"),
        F.col("vessel.vesselCommercialStatus").alias("st"),
        F.col("vessel.vesselName").alias("name"),
        F.col("vessel.vesselTeu").alias("teu"),
        F.col("vessel.vesselBuiltYear").alias("built"),
        F.col("vessel.vesselDwt").alias("dwt"),
        # Empty / whitespace-only vesselFlag → NULL (matches old Talend behavior)
        F.when(F.col("vessel.vesselFlag").isNull() | (F.trim(F.col("vessel.vesselFlag")) == ""), F.lit(None).cast("string"))
         .when(F.col("alphaliner_flag").isNull() & F.col("flag").isNull(), F.col("vessel.vesselFlag"))
         .when(F.col("alphaliner_flag").isNull() & F.col("flag").isNotNull(), F.col("flag"))
         .when(F.col("alphaliner_flag").isNotNull(), F.col("alphaliner_flag"))
         .otherwise(F.lit(None))
         .alias("flag"),
        F.col("vessel.vesselTeu14").alias("at14_t"),
        F.col("vessel.vesselReefPlugsNumber").alias("plugs"),
        F.col("vessel.vesselSpeedLaden").alias("speed"),
        F.col("vessel.vesselConsumption").alias("cns"),
        F.when(F.col("vessel.vesselCells") == "Y", 1).otherwise(0).cast("int").alias("c"),
        F.when(F.col("vessel.vesselGeared") == "Y", 1).otherwise(0).cast("int").alias("g"),
        F.col("vessel.vesselIceLevel").alias("ice"),
        F.col("vessel.vesselShipowner").alias("managing_owner"),
        F.lit(None).cast("string").alias("t"),
        F.lit(None).cast("string").alias("open_position"),
        F.col("vessel.DualFuel").alias("dual_fuel"),
        F.concat(F.col("vessel.engineDesign"), F.lit(' - '), F.col("vessel.engineModel")).alias("main_engine"),
        F.lit(None).cast("string").alias("open_start"),
        F.lit(None).cast("string").alias("open_end"),
        F.col("vessel.vesselDesign").alias("design_class"),
        F.col("vessel.vesselRoute").alias("route"),
        F.lit(None).cast("string").alias("pos_amend"),
        F.col("vessel.vesselOperator").alias("operator"),
        F.col("vessel.vesselBuilder").alias("ship_yard"),
        F.col("vessel.vesselBeam").alias("beam"),
        F.col("vessel.vesselLoa").alias("loa"),
        F.col("vessel.vesselDraft").alias("draft"),
        F.lit(None).cast("string").alias("owned_chartered"),
        F.col("vessel.vesselImo").alias("imo"),
        F.when(F.col("vessel.Scrubbers") == "Y", 1).otherwise(0).cast("int").alias("scrubber"),
        F.col("vessel.vesselGrt").alias("gt"),
        F.lit("API").alias("FileName"),
        F.col("vessel.vesselType").alias("vesselType"),
        F.concat_ws("-", F.col("meta.page_no").cast("string"), F.col("total_pages").cast("string")).alias("APIPageNum"),
        F.current_timestamp().alias("_ingestion_ts"),
    )

    # 5. Write to silver: full refresh or first run -> truncate + insert; otherwise -> merge on imo
    table_exists = spark.catalog.tableExists(TARGET_SILVER)
    if FULL_REFRESH or not table_exists:
        (
            silver_df
            .write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(TARGET_SILVER)
        )
        log.info(f"Silver: truncated and reloaded -> {TARGET_SILVER}")
    else:
        silver_delta = DeltaTable.forName(spark, TARGET_SILVER)
        (
            silver_delta.alias("t")
            .merge(silver_df.alias("s"), "t.imo = s.imo")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        log.info(f"Silver: merged on imo -> {TARGET_SILVER}")

    # 6. Mark bronze rows as normalized
    bronze_delta = DeltaTable.forName(spark, TARGET_BRONZE)
    bronze_delta.update(
        condition=(
            (F.col("_is_normalized") == False) 
        ),
        set={"_is_normalized": F.lit(True)},
    )
    log.info(f"Bronze: marked as normalized.")

In [0]:
if __name__ == "__main__":
    # 1. Resolve modifiedDate: last successful run from logs, or config default
    last_dt       = get_last_run_date(LOGS_TABLE, AXS_DEFAULT_DATE, FULL_REFRESH)
    modified_date = last_dt.strftime("%Y-%m-%d %H:%M:%S")
    # requests auto-encodes this in the URL as: YYYY-MM-DD%20HH%3AMM%3ASS
    log.info(f"modifiedDate (pre-encode): {modified_date}")

    # 2. Run full paginated pipeline
    run_pipeline(modified_date)
    normalize_to_silver()